# RSNA Knee Abnormality Detection: Shared Submission Inference

This version-independent submission notebook loads one or more checkpoints from a
selected model version, rebuilds the matching test MRI inputs, ensembles fold
probabilities, and writes `/kaggle/working/submission.csv`.

Set `cfg.model_version` to the version being submitted, for example `v01`, `v02`, or `v03`.
The attached Kaggle Dataset must contain checkpoints named
`<version>_fold_<fold>_best.pt`. V01, V02, and V03 share the current 3-plane 2.5D
EfficientNet architecture. Future model families should add a version-aware model
builder in this shared notebook.

Internet access is not required because each checkpoint contains the complete
backbone and classification-head weights.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import pickle
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

LABELS = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's",
    'Contusion',
    'Fracture',
]
PLANES = ('Sagittal', 'Coronal', 'Axial')
UID = 'StudyInstanceUID'


@dataclass
class InferenceCFG:
    model_version: str = 'v03'
    checkpoint_pattern: Optional[str] = None
    model_dataset_hint: Optional[str] = None
    batch_size: int = 1
    num_workers: int = 2
    seed: int = 2026


cfg = InferenceCFG()


def seed_everything(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(cfg.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
WORK_DIR = Path('/kaggle/working/rsna_knee_submission')
if not Path('/kaggle/working').exists():
    WORK_DIR = Path.cwd() / 'working' / 'rsna_knee_submission'
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Working directory:', WORK_DIR)




## 1. Locate the competition data and versioned checkpoints

The notebook searches Kaggle inputs for the official test CSV files. By default it
loads every checkpoint matching `<model_version>_fold_*_best.pt`, so folds from one
version are ensembled without mixing model versions. Set `cfg.checkpoint_pattern`
only when a custom naming pattern is required.


In [ ]:
def find_competition_root() -> Path:
    candidates = [
        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        Path('/kaggle/input/rsna-knee-abnormality-detection'),
        Path.cwd(),
        Path.cwd() / 'data',
    ]
    for root in candidates:
        if (root / 'test.csv').exists() and (root / 'test_series.csv').exists():
            return root

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for csv_path in kaggle_input.rglob('test_series.csv'):
            root = csv_path.parent
            if (root / 'test.csv').exists() and (root / 'sample_submission.csv').exists():
                return root

    raise FileNotFoundError(
        'Competition data was not found. Attach the RSNA Knee Abnormality Detection '
        'competition data to this notebook.'
    )


def find_checkpoints(pattern: str, dataset_hint: Optional[str]) -> List[Path]:
    if dataset_hint:
        search_root = Path(dataset_hint)
        if not search_root.exists():
            raise FileNotFoundError(f'Model dataset hint does not exist: {search_root}')
    else:
        search_root = Path('/kaggle/input')
        if not search_root.exists():
            search_root = Path.cwd()

    paths = sorted(path for path in search_root.rglob(pattern) if path.is_file())
    if not paths:
        raise FileNotFoundError(
            f'No checkpoint matching {pattern!r} was found under {search_root}. '
            'Attach a Kaggle Dataset containing the selected version checkpoints.'
        )
    return paths


def load_checkpoint(path: Path, map_location: object = 'cpu') -> Dict[str, object]:
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception as error:
        print(f'Safe checkpoint loading failed for {path.name}: {error}')
        print('Falling back to trusted-checkpoint loading. Use only checkpoints you created.')
        return torch.load(path, map_location=map_location, weights_only=False)


INPUT_ROOT = find_competition_root()
TEST_IMAGE_ROOT = INPUT_ROOT / 'test_series'
CHECKPOINT_PATTERN = cfg.checkpoint_pattern or f'{cfg.model_version}_fold_*_best.pt'
CHECKPOINT_PATHS = find_checkpoints(CHECKPOINT_PATTERN, cfg.model_dataset_hint)

test_df = pd.read_csv(INPUT_ROOT / 'test.csv')
test_series_df = pd.read_csv(INPUT_ROOT / 'test_series.csv')
sample_submission = pd.read_csv(INPUT_ROOT / 'sample_submission.csv')

assert test_df[UID].is_unique, 'StudyInstanceUID must be unique in test.csv'
assert set(sample_submission.columns) == {UID, *LABELS}
assert TEST_IMAGE_ROOT.exists(), f'Test image directory does not exist: {TEST_IMAGE_ROOT}'

first_checkpoint = load_checkpoint(CHECKPOINT_PATHS[0])
MODEL_CONFIG = first_checkpoint.get('config', {})
checkpoint_version = MODEL_CONFIG.get('version', first_checkpoint.get('version'))
if checkpoint_version is not None and str(checkpoint_version).lower() != cfg.model_version.lower():
    raise ValueError(
        f'Checkpoint version {checkpoint_version!r} does not match '
        f'cfg.model_version={cfg.model_version!r}'
    )
checkpoint_labels = first_checkpoint.get('labels', LABELS)
assert list(checkpoint_labels) == LABELS, 'Checkpoint label order does not match the competition schema'

IMAGE_SIZE = int(MODEL_CONFIG.get('image_size', 320))
SAMPLES_PER_PLANE = int(MODEL_CONFIG.get('samples_per_plane', 6))
TRIPLET_GAP = int(MODEL_CONFIG.get('triplet_gap', 2))
BACKBONE = str(MODEL_CONFIG.get('backbone', 'efficientnet_b0'))
DROPOUT = float(MODEL_CONFIG.get('dropout', 0.30))

del first_checkpoint
gc.collect()

print('Model version:', cfg.model_version)
print('Checkpoint pattern:', CHECKPOINT_PATTERN)
print('Competition root:', INPUT_ROOT)
print('Test studies / series:', len(test_df), len(test_series_df))
print('Checkpoints:')
for checkpoint_path in CHECKPOINT_PATHS:
    print(' -', checkpoint_path)
print('Model configuration:', json.dumps({
    'image_size': IMAGE_SIZE,
    'samples_per_plane': SAMPLES_PER_PLANE,
    'triplet_gap': TRIPLET_GAP,
    'backbone': BACKBONE,
    'dropout': DROPOUT,
}, indent=2))




## 2. Select one primary series per anatomical plane

Selection follows the current 3-plane 2.5D model family: prefer a fluid-sensitive,
fat-suppressed series, then use another series in the same plane. Within the same
metadata priority, prefer a slice count close to 32.




In [ ]:
def count_dicom_files(series_dir: Path) -> int:
    if not series_dir.exists():
        return 0
    return sum(
        1 for path in series_dir.iterdir()
        if path.is_file() and path.suffix.lower() == '.dcm'
    )


def build_selected_series(
    studies: pd.DataFrame,
    series_frame: pd.DataFrame,
    image_root: Path,
) -> pd.DataFrame:
    grouped = {key: group for key, group in series_frame.groupby(UID, sort=False)}
    selected_rows = []

    for study_uid in tqdm(studies[UID].astype(str), desc='Select test series'):
        study_series = grouped.get(study_uid)
        record = {UID: study_uid}

        for plane in PLANES:
            candidates = [] if study_series is None else study_series[
                study_series['Anatomical_Plane'].eq(plane)
            ].to_dict('records')

            scored = []
            for candidate in candidates:
                series_uid = str(candidate['SeriesInstanceUID'])
                n_slices = count_dicom_files(image_root / study_uid / series_uid)
                fluid = int(candidate.get('Fluid_Sensitive', 0))
                fat = int(candidate.get('Fat_Suppression', 0))
                both = int(fluid == 1 and fat == 1)
                score = (
                    int(n_slices > 0),
                    both,
                    fluid,
                    fat,
                    -abs(n_slices - 32),
                    n_slices,
                    series_uid,
                )
                scored.append((score, candidate, n_slices))

            if scored:
                _, best, n_slices = max(scored, key=lambda item: item[0])
                record[f'{plane}_SeriesInstanceUID'] = str(best['SeriesInstanceUID'])
                record[f'{plane}_n_slices'] = int(n_slices)
                record[f'{plane}_Fluid_Sensitive'] = int(best.get('Fluid_Sensitive', 0))
                record[f'{plane}_Fat_Suppression'] = int(best.get('Fat_Suppression', 0))
            else:
                record[f'{plane}_SeriesInstanceUID'] = ''
                record[f'{plane}_n_slices'] = 0
                record[f'{plane}_Fluid_Sensitive'] = 0
                record[f'{plane}_Fat_Suppression'] = 0

        selected_rows.append(record)

    return pd.DataFrame(selected_rows)


test_selected = build_selected_series(test_df, test_series_df, TEST_IMAGE_ROOT)
test_selected.to_csv(WORK_DIR / 'test_selected_series.csv', index=False)

coverage_rows = []
for plane in PLANES:
    coverage_rows.append({
        'plane': plane,
        'available': int((test_selected[f'{plane}_n_slices'] > 0).sum()),
        'preferred_fs': int(
            (
                (test_selected[f'{plane}_Fluid_Sensitive'] == 1)
                & (test_selected[f'{plane}_Fat_Suppression'] == 1)
            ).sum()
        ),
        'median_slices': float(test_selected[f'{plane}_n_slices'].median()),
    })

display(pd.DataFrame(coverage_rows))
display(test_selected.head())



## 3. Sort DICOM slices

Spatial sorting uses the projection of `ImagePositionPatient` onto the slice
normal derived from `ImageOrientationPatient`. `InstanceNumber` and filename
order are deterministic fallbacks.



In [ ]:
ORDER_TAGS = ['ImageOrientationPatient', 'ImagePositionPatient', 'InstanceNumber']


def safe_float_list(value: object, expected: int) -> Optional[np.ndarray]:
    try:
        array = np.asarray([float(item) for item in value], dtype=np.float64)
        return array if len(array) == expected else None
    except Exception:
        return None


def sort_dicom_files(series_dir: Path) -> Tuple[List[str], str, int]:
    files = sorted(
        path for path in series_dir.iterdir()
        if path.is_file() and path.suffix.lower() == '.dcm'
    ) if series_dir.exists() else []
    if not files:
        return [], 'empty', 0

    records = []
    header_errors = 0
    for path in files:
        position_value = None
        instance_value = None
        try:
            ds = pydicom.dcmread(
                str(path),
                stop_before_pixels=True,
                force=True,
                specific_tags=ORDER_TAGS,
            )
            orientation = safe_float_list(getattr(ds, 'ImageOrientationPatient', None), 6)
            position = safe_float_list(getattr(ds, 'ImagePositionPatient', None), 3)
            if orientation is not None and position is not None:
                normal = np.cross(orientation[:3], orientation[3:])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    position_value = float(np.dot(position, normal / norm))
            if getattr(ds, 'InstanceNumber', None) is not None:
                instance_value = float(ds.InstanceNumber)
        except Exception:
            header_errors += 1
        records.append((path, position_value, instance_value))

    positions = [record[1] for record in records]
    instances = [record[2] for record in records]
    if all(value is not None for value in positions) and len(set(positions)) == len(records):
        records.sort(key=lambda item: item[1])
        method = 'ImagePositionPatient'
    elif all(value is not None for value in instances) and len(set(instances)) == len(records):
        records.sort(key=lambda item: item[2])
        method = 'InstanceNumber'
    else:
        records.sort(key=lambda item: (
            item[2] is None,
            item[2] if item[2] is not None else math.inf,
            item[0].name,
        ))
        method = 'mixed_fallback'

    return [str(record[0]) for record in records], method, header_errors


def build_ordered_paths(
    selected: pd.DataFrame,
    image_root: Path,
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    ordered_paths: Dict[str, List[str]] = {}
    audit_rows = []

    for _, row in tqdm(
        selected.iterrows(),
        total=len(selected),
        desc='Order test DICOM',
    ):
        study_uid = str(row[UID])
        for plane in PLANES:
            value = row[f'{plane}_SeriesInstanceUID']
            if pd.isna(value) or not str(value):
                continue
            series_uid = str(value)
            if series_uid in ordered_paths:
                continue

            paths, method, errors = sort_dicom_files(
                image_root / study_uid / series_uid
            )
            ordered_paths[series_uid] = paths
            audit_rows.append({
                UID: study_uid,
                'SeriesInstanceUID': series_uid,
                'plane': plane,
                'n_slices': len(paths),
                'sort_method': method,
                'header_errors': errors,
            })

    return ordered_paths, pd.DataFrame(audit_rows)


test_ordered_paths, test_order_audit = build_ordered_paths(
    test_selected,
    TEST_IMAGE_ROOT,
)
test_order_audit.to_csv(WORK_DIR / 'test_ordering_audit.csv', index=False)
with open(WORK_DIR / 'test_ordered_paths.pkl', 'wb') as file:
    pickle.dump(test_ordered_paths, file, protocol=pickle.HIGHEST_PROTOCOL)

print('Sorting methods:')
display(test_order_audit['sort_method'].value_counts(dropna=False).to_frame('series'))
print('Header errors:', int(test_order_audit['header_errors'].sum()))
print('Empty selected series:', int((test_order_audit['n_slices'] == 0).sum()))



## 4. DICOM preprocessing and 2.5D test dataset

Each plane is sampled uniformly at the number of positions stored in the training
checkpoint. Every center position creates a three-channel
`[n-gap, n, n+gap]` input with the same clipping and normalization used in training.



In [ ]:
def read_dicom_pixels(path: str) -> Optional[np.ndarray]:
    try:
        ds = pydicom.dcmread(path, force=True)
        image = np.asarray(ds.pixel_array)
        image = np.squeeze(image)
        while image.ndim > 2:
            image = image[image.shape[0] // 2]
        if image.ndim != 2:
            return None

        image = image.astype(np.float32)
        slope = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
        intercept = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
        image = image * slope + intercept
        if str(getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2')) == 'MONOCHROME1':
            image = image.max() + image.min() - image
        return image
    except Exception:
        return None


def sample_centers(n_slices: int, count: int) -> np.ndarray:
    if n_slices <= 1:
        return np.zeros(count, dtype=np.int64)
    return np.rint(
        np.linspace(0, n_slices - 1, count + 2)[1:-1]
    ).astype(np.int64)


def normalize_triplet(
    images: Sequence[Optional[np.ndarray]],
    size: int,
) -> np.ndarray:
    valid = [image for image in images if image is not None and image.size > 0]
    if not valid:
        return np.zeros((size, size, 3), dtype=np.float32)

    values = np.concatenate([image.reshape(-1) for image in valid])
    low, high = np.percentile(values, [1.0, 99.0])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low = float(np.min(values))
        high = float(np.max(values))
    if high <= low:
        high = low + 1.0

    fallback_shape = valid[0].shape
    channels = []
    for image in images:
        if image is None:
            image = np.zeros(fallback_shape, dtype=np.float32)
        image = np.clip(image, low, high)
        image = (image - low) / (high - low)
        image = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
        channels.append(image.astype(np.float32))
    return np.stack(channels, axis=-1)


def preflight_dicom_check(
    path_cache: Dict[str, List[str]],
    maximum_files: int = 24,
) -> None:
    candidates = [paths[len(paths) // 2] for paths in path_cache.values() if paths]
    rng = random.Random(cfg.seed)
    rng.shuffle(candidates)
    candidates = candidates[:min(maximum_files, len(candidates))]
    failures = sum(
        read_dicom_pixels(path) is None
        for path in tqdm(candidates, desc='DICOM preflight')
    )
    print(f'DICOM preflight failures: {failures}/{len(candidates)}')
    if candidates and failures / len(candidates) > 0.10:
        raise RuntimeError(
            'More than 10% of sampled DICOM files could not be decoded. '
            'Check pydicom decoding plugins and competition data.'
        )


preflight_dicom_check(test_ordered_paths)

IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)


class KneeTestDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        selected: pd.DataFrame,
        ordered_paths: Dict[str, List[str]],
    ) -> None:
        self.frame = frame.reset_index(drop=True).copy()
        self.selected = selected.set_index(UID)
        self.ordered_paths = ordered_paths

    def __len__(self) -> int:
        return len(self.frame)

    def _load_plane(self, paths: List[str]) -> torch.Tensor:
        output = torch.zeros(
            SAMPLES_PER_PLANE,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
            dtype=torch.float32,
        )
        if not paths:
            return output

        centers = sample_centers(len(paths), SAMPLES_PER_PLANE)
        pixel_cache: Dict[str, Optional[np.ndarray]] = {}
        for sample_index, center in enumerate(centers):
            indices = np.clip(
                [center - TRIPLET_GAP, center, center + TRIPLET_GAP],
                0,
                len(paths) - 1,
            )
            images = []
            for index in indices:
                path = paths[int(index)]
                if path not in pixel_cache:
                    pixel_cache[path] = read_dicom_pixels(path)
                images.append(pixel_cache[path])

            triplet = normalize_triplet(images, IMAGE_SIZE)
            triplet = (triplet - IMAGENET_MEAN) / IMAGENET_STD
            output[sample_index] = torch.from_numpy(
                triplet.transpose(2, 0, 1).copy()
            )
        return output

    def __getitem__(self, index: int) -> Dict[str, object]:
        row = self.frame.iloc[index]
        study_uid = str(row[UID])
        selected_row = self.selected.loc[study_uid]

        plane_images = []
        plane_mask = []
        sequence_meta = []
        for plane in PLANES:
            value = selected_row[f'{plane}_SeriesInstanceUID']
            series_uid = '' if pd.isna(value) else str(value)
            paths = self.ordered_paths.get(series_uid, []) if series_uid else []
            valid = float(len(paths) > 0)
            plane_images.append(self._load_plane(paths))
            plane_mask.append(valid)
            sequence_meta.append([
                float(selected_row[f'{plane}_Fluid_Sensitive']) if valid else 0.0,
                float(selected_row[f'{plane}_Fat_Suppression']) if valid else 0.0,
            ])

        return {
            UID: study_uid,
            'images': torch.stack(plane_images),
            'plane_mask': torch.tensor(plane_mask, dtype=torch.float32),
            'sequence_meta': torch.tensor(sequence_meta, dtype=torch.float32),
        }


test_dataset = KneeTestDataset(
    test_df,
    test_selected,
    test_ordered_paths,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=AMP_ENABLED,
    persistent_workers=cfg.num_workers > 0,
    drop_last=False,
)

sample_batch = next(iter(test_loader))
print('Image batch shape:', tuple(sample_batch['images'].shape))
print('Plane masks:', sample_batch['plane_mask'])
assert sample_batch['images'].shape[1:] == (
    len(PLANES),
    SAMPLES_PER_PLANE,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
)



## 5. Recreate the trained model

The model is created without external pretrained weights. Every parameter,
including the EfficientNet backbone, is restored from the trusted fold checkpoint.



In [ ]:
class MultiPlaneEfficientNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        try:
            import timm
        except ImportError as error:
            raise ImportError(
                'The Kaggle environment must provide timm to recreate the trained model.'
            ) from error

        self.backbone = timm.create_model(
            BACKBONE,
            pretrained=False,
            num_classes=0,
            global_pool='avg',
        )
        self.feature_dim = int(self.backbone.num_features)

        metadata_dim = len(PLANES) * 3
        fusion_dim = self.feature_dim * len(PLANES) + metadata_dim
        hidden_dim = 512
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, len(LABELS)),
        )

    def forward(
        self,
        images: torch.Tensor,
        plane_mask: torch.Tensor,
        sequence_meta: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, n_planes, n_samples, channels, height, width = images.shape
        images = images.reshape(
            batch_size * n_planes * n_samples,
            channels,
            height,
            width,
        )
        features = self.backbone(images)
        if features.ndim > 2:
            features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        features = features.reshape(
            batch_size,
            n_planes,
            n_samples,
            self.feature_dim,
        )
        plane_features = features.mean(dim=2) * plane_mask.unsqueeze(-1)

        metadata = torch.cat(
            [plane_mask.unsqueeze(-1), sequence_meta],
            dim=-1,
        ).flatten(1)
        fused = torch.cat([plane_features.flatten(1), metadata], dim=1)
        return self.head(fused)


def validate_checkpoint(checkpoint: Dict[str, object], path: Path) -> None:
    labels = list(checkpoint.get('labels', LABELS))
    if labels != LABELS:
        raise ValueError(f'Label order mismatch in checkpoint: {path}')

    config = checkpoint.get('config', {})
    version = config.get('version', checkpoint.get('version'))
    if version is not None and str(version).lower() != cfg.model_version.lower():
        raise ValueError(
            f'Checkpoint version mismatch: {version!r} != {cfg.model_version!r} in {path}'
        )
    expected = {
        'image_size': IMAGE_SIZE,
        'samples_per_plane': SAMPLES_PER_PLANE,
        'triplet_gap': TRIPLET_GAP,
        'backbone': BACKBONE,
    }
    for key, expected_value in expected.items():
        value = config.get(key, expected_value)
        if value != expected_value:
            raise ValueError(
                f'Checkpoint configuration mismatch for {key}: '
                f'{value!r} != {expected_value!r} in {path}'
            )


model_preview = MultiPlaneEfficientNet()
parameter_count = sum(parameter.numel() for parameter in model_preview.parameters())
print(f'Model parameters: {parameter_count:,}')
del model_preview
gc.collect()




## 6. Run checkpoint inference and ensemble folds



In [ ]:
@torch.no_grad()
def predict(
    model: nn.Module,
    loader: DataLoader,
) -> Tuple[np.ndarray, List[str]]:
    model.eval()
    predictions = []
    study_ids = []

    for batch in tqdm(loader, desc='Test inference'):
        images = batch['images'].to(DEVICE, non_blocking=True)
        plane_mask = batch['plane_mask'].to(DEVICE, non_blocking=True)
        sequence_meta = batch['sequence_meta'].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=AMP_ENABLED,
        ):
            logits = model(images, plane_mask, sequence_meta)
        probabilities = torch.sigmoid(logits.float())
        predictions.append(probabilities.cpu().numpy())
        study_ids.extend(list(batch[UID]))

    return np.concatenate(predictions), study_ids


checkpoint_predictions = []
inference_ids: Optional[List[str]] = None

for checkpoint_path in CHECKPOINT_PATHS:
    print('Loading:', checkpoint_path)
    checkpoint = load_checkpoint(checkpoint_path)
    validate_checkpoint(checkpoint, checkpoint_path)

    model = MultiPlaneEfficientNet()
    model.load_state_dict(checkpoint['model'], strict=True)
    model = model.to(DEVICE)

    probabilities, current_ids = predict(model, test_loader)
    if inference_ids is None:
        inference_ids = current_ids
    elif inference_ids != current_ids:
        raise RuntimeError('Study order changed between checkpoint inference runs')

    checkpoint_predictions.append(probabilities.astype(np.float32))
    del checkpoint, model, probabilities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert inference_ids is not None
ensemble_probability = np.mean(checkpoint_predictions, axis=0, dtype=np.float32)
assert ensemble_probability.shape == (len(test_df), len(LABELS))

print('Ensembled checkpoints:', len(checkpoint_predictions))
print('Prediction range:', float(ensemble_probability.min()), float(ensemble_probability.max()))



## 7. Build and validate submission.csv



In [ ]:
prediction_frame = pd.DataFrame(ensemble_probability, columns=LABELS)
prediction_frame.insert(0, UID, inference_ids)

submission = sample_submission[[UID]].merge(
    prediction_frame,
    on=UID,
    how='left',
    validate='one_to_one',
)
submission = submission[[UID, *LABELS]]

if submission[LABELS].isna().any().any():
    missing_studies = submission.loc[
        submission[LABELS].isna().any(axis=1),
        UID,
    ].tolist()
    raise RuntimeError(f'Missing predictions for test studies: {missing_studies[:5]}')

submission[LABELS] = submission[LABELS].clip(1e-6, 1 - 1e-6)
if not np.isfinite(submission[LABELS].to_numpy()).all():
    raise RuntimeError('Submission contains non-finite probabilities')
if not submission[LABELS].to_numpy().min() >= 0:
    raise RuntimeError('Submission contains probabilities below zero')
if not submission[LABELS].to_numpy().max() <= 1:
    raise RuntimeError('Submission contains probabilities above one')

submission_path = Path('/kaggle/working/submission.csv')
if not submission_path.parent.exists():
    submission_path = WORK_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
submission.to_csv(WORK_DIR / 'submission.csv', index=False)

print('Submission path:', submission_path)
print('Submission shape:', submission.shape)
print('Missing values:', int(submission.isna().sum().sum()))
display(submission.head())



## 8. Submit the file

After the notebook completes:

1. Open the Kaggle Notebook output panel.
2. Confirm that `submission.csv` has the expected row and column counts.
3. Submit `/kaggle/working/submission.csv` to the competition.
4. Record the public score together with the checkpoint name and notebook version.

The notebook automatically ensembles all attached folds matching the selected model version.
Record both `cfg.model_version` and the checkpoint filenames with every submission.

